# Feature Engineering — FIFA World Cup 2026 Player Performance

No `01_eda.ipynb` exploramos os dados e tomamos algumas decisões importantes:

- **Alvo:** regressão em `player_rating`.
- **Escopo:** filtrar para `minutes_played > 0` — 42% das linhas são jogadores que não entraram em campo, onde a nota é trivialmente 0.
- **Vazamento de dados confirmado:** `performance_score` (e, por extensão, `tournament_rating`, quase idêntico ao alvo) ficam fora das features.
- **Falso alarme:** `top_speed_kmh` parecia vazamento (corr. 0.98) mas era um artefato das linhas sem minutos jogados — não tem sinal real, fica de fora por ser inútil, não por vazar informação.
- **Ruído sem confiabilidade:** os totais acumulados de torneio (`total_goals_tournament` etc.) não têm coerência temporal — ficam de fora.

**Objetivo deste notebook:** partir dessas decisões, montar de fato o conjunto de features (selecionar colunas, tratar categóricas de alta cardinalidade) e deixar os dados prontos para a etapa de modelagem.

## 1. Importando as ferramentas

Começamos só com pandas, seaborn e matplotlib — as mesmas do notebook anterior, pra manipular os dados e visualizar as features conforme formos construindo. Ferramentas específicas de encoding (scikit-learn) a gente importa mais adiante, no momento em que forem realmente usadas.

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## 2. Carregando os dados e aplicando o escopo

Cada notebook parte do CSV original (não temos módulos `.py` compartilhados de propósito — o projeto é só notebooks). A primeira coisa a fazer é reaplicar a decisão de escopo do notebook anterior: **remover as linhas de jogadores que não entraram em campo** (`minutes_played == 0`), já que prever nota 0 pra quem não jogou é trivial e não é o problema que queremos resolver.

In [2]:
data_df = pd.read_csv("../data/fifa_world_cup_2026_player_performance.csv")

played_df = data_df[data_df["minutes_played"] > 0].copy()

print(f"Linhas antes do filtro: {len(data_df)}")
print(f"Linhas depois do filtro (minutes_played > 0): {len(played_df)}")

Linhas antes do filtro: 54600
Linhas depois do filtro (minutes_played > 0): 31558


## 3. Selecionando as colunas candidatas

Com base na conclusão do `01_eda`, montamos a lista do que **sai** do conjunto de features:

- Identificadores: `player_id`, `player_name`, `match_id`
- Vazamento: `performance_score`, `tournament_rating`
- Sem sinal real: `top_speed_kmh`
- Ruído sem coerência temporal: `total_goals_tournament`, `total_assists_tournament`, `total_minutes_tournament`
- Outro resultado pós-partida (não uma causa do rating): `player_of_match_awards`
- O próprio alvo: `player_rating` (guardamos à parte, não é uma feature)

Tudo o que sobra é candidato a feature.

In [3]:
excluded_cols = [
    # identificadores
    "player_id", "player_name", "match_id",
    # vazamento de dados
    "performance_score", "tournament_rating",
    # sem sinal real (artefato investigado no notebook anterior)
    "top_speed_kmh",
    # ruído sem coerência temporal
    "total_goals_tournament", "total_assists_tournament", "total_minutes_tournament",
    # outro resultado pós-partida, não uma causa do rating
    "player_of_match_awards",
]

target_col = "player_rating"

feature_cols = [c for c in played_df.columns if c not in excluded_cols + [target_col]]

print(f"Features candidatas: {len(feature_cols)}")
feature_cols

Features candidatas: 64


['age',
 'nationality',
 'team',
 'jersey_number',
 'position',
 'height_cm',
 'weight_kg',
 'preferred_foot',
 'club_name',
 'market_value_eur',
 'match_date',
 'stadium',
 'city',
 'opponent_team',
 'tournament_stage',
 'match_result',
 'goals_team',
 'goals_opponent',
 'minutes_played',
 'goals',
 'assists',
 'shots',
 'shots_on_target',
 'expected_goals_xg',
 'expected_assists_xa',
 'key_passes',
 'successful_passes',
 'total_passes',
 'pass_accuracy',
 'dribbles_attempted',
 'successful_dribbles',
 'crosses',
 'successful_crosses',
 'tackles',
 'interceptions',
 'clearances',
 'blocks',
 'aerial_duels_won',
 'aerial_duels_lost',
 'recoveries',
 'defensive_actions',
 'fouls_committed',
 'fouls_suffered',
 'yellow_cards',
 'red_cards',
 'offsides',
 'saves',
 'save_percentage',
 'punches',
 'clean_sheet',
 'goals_conceded',
 'penalty_saves',
 'distance_covered_km',
 'sprint_distance_km',
 'accelerations',
 'decelerations',
 'stamina_score',
 'offensive_contribution',
 'defensive_con

## 4. Cardinalidade das colunas categóricas

Modelos de ML não entendem texto — colunas categóricas (`position`, `team`, etc.) precisam virar números antes de entrar no modelo. A técnica mais simples é o **one-hot encoding** (uma coluna binária por categoria), mas ela só funciona bem quando a coluna tem poucas categorias — com muitas, o número de colunas explode e a maioria fica quase sempre zero.

Por isso, antes de decidir a técnica, precisamos saber: quantas categorias cada coluna tem?

In [4]:
categorical_cols = played_df[feature_cols].select_dtypes(include="str").columns.tolist()

played_df[categorical_cols].nunique().sort_values()

preferred_foot        2
match_result          3
position              4
tournament_stage      7
stadium              16
city                 16
nationality          48
team                 48
opponent_team        48
match_date           51
club_name           122
dtype: int64

Dá pra dividir em três grupos:

- **Baixa cardinalidade (2 a 16 categorias):** `preferred_foot`, `match_result`, `position`, `tournament_stage`, `stadium`, `city` — one-hot encoding direto, sem problema.
- **Cardinalidade média (48):** `nationality`, `team`, `opponent_team` — one-hot ainda é viável (48 colunas novas não é absurdo com 31.558 linhas), mas vale checar redundância antes.
- **Alta cardinalidade:** `club_name` (122) e `match_date` (51) — one-hot aqui geraria muitas colunas esparsas; precisam de uma abordagem diferente.

Antes de decidir a técnica, um detalhe chamou atenção: `nationality` e `team` têm exatamente a mesma cardinalidade (48). Pode ser coincidência, ou pode ser a mesma informação escrita de duas formas (ex: "Spanish" vs "Spain"). Vale checar.

In [5]:
nationality_team_map = played_df[["nationality", "team"]].drop_duplicates()

print(f"Combinações únicas de (nationality, team): {len(nationality_team_map)}")
nationality_team_map.head()

Combinações únicas de (nationality, team): 48


,nationality,team
0,Spanish,Spain
26,South African,South Africa
52,Peruvian,Peru
78,Dutch,Netherlands
104,Moroccan,Morocco


Confirmado: 48 combinações únicas para 48 categorias em cada coluna — é uma relação 1 para 1. `nationality` é só o gentílico de `team` ("Spanish" → "Spain"). São a mesma informação escrita de duas formas; `nationality` sai do conjunto de features (redundante com `team`).

E a tabela de cardinalidade mostra outro par com o mesmo padrão suspeito: `stadium` e `city`, ambas com 16 categorias — a Copa de 2026 tem 16 sedes, e cada estádio fica em uma cidade. Se a relação for 1 pra 1 de novo, uma das duas é redundante. Mesmo teste:

In [6]:
stadium_city_map = played_df[["stadium", "city"]].drop_duplicates()

print(f"Combinações únicas de (stadium, city): {len(stadium_city_map)}")
stadium_city_map.head()

Combinações únicas de (stadium, city): 16


,stadium,city
0,Hard Rock Stadium,Miami
52,Arrowhead Stadium,Kansas City
104,Gillette Stadium,Boston
156,Lincoln Financial Field,Philadelphia
260,Estadio Azteca,Mexico City


Mesmo caso: 16 combinações para 16 categorias — cada estádio pertence a exatamente uma cidade, as duas colunas carregam a mesma informação. Mantemos `stadium` (identifica o local com mais precisão, caso uma cidade viesse a ter dois estádios) e descartamos `city`.

## 5. `match_date`: a data da partida faz sentido?

Sobraram as duas colunas de alta cardinalidade. Começando por `match_date` (51 datas distintas): antes de pensar em *como* codificar uma data, vale perguntar *o que ela deveria informar*. Em um torneio real, a data carrega a estrutura da competição — fase de grupos primeiro, final por último — e efeitos como desgaste acumulado. Ou seja, se a data tem sinal de verdade, ele deveria aparecer alinhado com `tournament_stage`.

No `01_eda` já pegamos os totais acumulados de torneio sem coerência temporal. Vamos aplicar o mesmo teste de sanidade aqui: cruzar as datas com as fases do torneio.

In [7]:
stage_order = [
    "Group Stage", "Round of 32", "Round of 16",
    "Quarter Finals", "Semi Finals", "Third Place Match", "Final",
]

played_df.groupby("tournament_stage")["match_date"].agg(["min", "max", "nunique"]).loc[stage_order]

,min,max,nunique
tournament_stage,,,
Group Stage,2026-06-11,2026-07-31,51
Round of 32,2026-06-11,2026-07-31,49
Round of 16,2026-06-11,2026-07-31,46
Quarter Finals,2026-06-11,2026-07-31,43
Semi Finals,2026-06-13,2026-07-29,28
Third Place Match,2026-06-12,2026-07-27,18
Final,2026-06-15,2026-07-30,16


Nenhuma coerência: a "Final" acontece em 16 datas diferentes, espalhadas pelo torneio inteiro, e a fase de grupos vai até o último dia. Em um Mundial real a final é *um* jogo, em *uma* data, depois de todas as outras fases. É o mesmo padrão dos totais acumulados de torneio: o gerador do dataset sorteou as datas sem respeitar a estrutura da competição.

Conclusão: `match_date` é ruído, não informação. Sai do conjunto de features — e de quebra nem precisamos discutir como codificar datas.

## 6. `club_name`: 122 categorias valem alguma coisa?

Última categórica pendente. One-hot em 122 categorias criaria 122 colunas esparsas — possível, mas ruim. Antes de escolher uma técnica mais sofisticada, a pergunta certa é anterior a isso: **o clube do jogador carrega algum sinal sobre o rating?**

A intuição de por que *poderia* carregar: clubes grandes concentram jogadores melhores. Como cada jogador pertence a um único clube, a média de rating de um clube reflete a qualidade do seu elenco (cada clube tem ~10 jogadores no dataset: 1.248 jogadores / 122 clubes).

Como medir: comparar o quanto as **médias por clube** variam entre si com o quanto o rating varia no geral. Se o clube não informasse nada, as médias de todos os clubes seriam quase iguais entre si (todas próximas da média global).

In [8]:
overall_std = played_df["player_rating"].std()
club_rating = played_df.groupby("club_name")["player_rating"].agg(["mean", "count"])

print(f"Desvio padrão global do rating: {overall_std:.3f}")
print(f"Desvio padrão das médias por clube: {club_rating['mean'].std():.3f}")
print(f"Médias por clube: de {club_rating['mean'].min():.2f} a {club_rating['mean'].max():.2f}")
print(f"Linhas por clube: mín {club_rating['count'].min()}, mediana {club_rating['count'].median():.0f}, máx {club_rating['count'].max()}")

Desvio padrão global do rating: 0.736
Desvio padrão das médias por clube: 0.123
Médias por clube: de 6.03 a 6.60
Linhas por clube: mín 47, mediana 245, máx 540


Como ler esses números:

- Se o clube não carregasse informação nenhuma, a média de um clube com ~245 linhas flutuaria por acaso em torno de ±0.05 da média global (0.736 / √245 ≈ 0.047).
- As médias observadas variam com desvio de **0.123** — mais que isso. Ou seja, **existe sinal**: clubes diferentes agrupam jogadores de níveis diferentes. (Uma nuance honesta: as ~245 linhas de um clube não são independentes — são os mesmos ~10 jogadores repetidos partida após partida. Parte dessa variação é simplesmente "quem joga lá", mas é exatamente isso que torna o clube um proxy útil da qualidade do elenco.)
- Mas é um sinal **fraco**: a diferença entre o "melhor" e o "pior" clube é de ~0.6 ponto de rating, contra um desvio global de 0.736. O clube ajuda, mas não decide.

Com isso, as opções para `club_name`:

1. **Descartar** — simples e seguro, mas joga fora um sinal real (ainda que fraco).
2. **One-hot** — não perde nada, ao custo de 122 colunas esparsas.
3. **Target encoding** — substituir cada clube pela média de rating dos seus jogadores. Compacta (1 coluna) e captura exatamente o sinal que medimos acima, **mas** usa o próprio alvo para construir a feature, o que exige cuidado redobrado contra vazamento (as médias precisam ser calculadas só no conjunto de treino, nunca no de teste).

*(Decisão em aberto — a discutir antes de seguir para a montagem final do conjunto de features.)*